In [0]:
from pyspark.sql.functions import (
    col, row_number, current_timestamp, sha2, concat_ws,
    expr, lit
)
from pyspark.sql.window import Window

prices_bronze_df = spark.table("dbr_dev_ua5816bd.roksolana_shendiu770_bronze.petroleum_prices_raw")

prices_cleaned_df = (prices_bronze_df
    .withColumnRenamed("series", "series_bk")
    .withColumnRenamed("product-name", "product_name")
    .withColumn("effective_from", expr("try_cast(period as date)"))
    .withColumn("price", expr("try_cast(value as decimal(10,3))"))
    .select(
        "series_bk", "product_name", "units", "price", "effective_from",
        "source_filename", "ingestion_timestamp"
    )
)

In [0]:
w_prices = Window.partitionBy("series_bk", "effective_from") \
                  .orderBy(col("ingestion_timestamp").desc())

prices_deduped_df = (prices_cleaned_df
    .withColumn("rn", row_number().over(w_prices))
    .filter(col("rn") == 1)
    .drop("rn")
)

In [0]:
prices_deduped_df.printSchema()
prices_deduped_df.count()

In [0]:
prices_final_df = prices_deduped_df.withColumn(
    "price_sk",
    sha2(concat_ws("||", 
        col("series_bk"), 
        col("effective_from").cast("string")
    ), 256)
).withColumn(
    "effective_to", lit(None).cast("date")
).withColumn(
    "is_current", lit(True)
).withColumn(
    "_source_system", lit("EIA_petroleum_prices")
).withColumn(
    "_ingested_at", current_timestamp()
).select(
    "price_sk", "series_bk", "product_name", "units", "price",
    "effective_from", "effective_to", "is_current",
    "_source_system", "_ingested_at"
)

prices_final_df.printSchema()

In [0]:
from delta.tables import DeltaTable

prices_target = DeltaTable.forName(
    spark, "dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_prices"
)

(prices_target.alias("t")
    .merge(
        prices_final_df.alias("s"),
        "t.series_bk = s.series_bk AND t.is_current = true"
    )
    .whenMatchedUpdate(
        condition="t.price <> s.price",
        set={
            "effective_to": "s.effective_from",
            "is_current": "false"
        }
    )
    .execute()
)

In [0]:
result_step1 = (prices_target.alias("t")
    .merge(
        prices_final_df.alias("s"),
        "t.series_bk = s.series_bk AND t.is_current = true"
    )
    .whenMatchedUpdate(
        condition="t.price <> s.price",
        set={
            "effective_to": "s.effective_from",
            "is_current": "false"
        }
    )
    .execute()
)

display(result_step1)

In [0]:
(prices_target.alias("t")
    .merge(
        prices_final_df.alias("s"),
        "t.series_bk = s.series_bk AND t.effective_from = s.effective_from"
    )
    .whenNotMatchedInsert(
        values={
            "price_sk": "s.price_sk",
            "series_bk": "s.series_bk",
            "product_name": "s.product_name",
            "units": "s.units",
            "price": "s.price",
            "effective_from": "s.effective_from",
            "effective_to": "s.effective_to",
            "is_current": "s.is_current",
            "_source_system": "s._source_system",
            "_ingested_at": "s._ingested_at"
        }
    )
    .execute()
)

In [0]:
result_step2 = (prices_target.alias("t")
    .merge(
        prices_final_df.alias("s"),
        "t.series_bk = s.series_bk AND t.effective_from = s.effective_from"
    )
    .whenNotMatchedInsert(
        values={
            "price_sk": "s.price_sk",
            "series_bk": "s.series_bk",
            "product_name": "s.product_name",
            "units": "s.units",
            "price": "s.price",
            "effective_from": "s.effective_from",
            "effective_to": "s.effective_to",
            "is_current": "s.is_current",
            "_source_system": "s._source_system",
            "_ingested_at": "s._ingested_at"
        }
    )
    .execute()
)

display(result_step2)

In [0]:
spark.sql("""
SELECT COUNT(*) AS row_count 
FROM dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_prices
""").show()

In [0]:
prices_final_df.count()

In [0]:
%sql
SELECT is_current, COUNT(*) 
FROM dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_prices
GROUP BY is_current